# 2.3 Προετοιμασία δεδομένων (FLORES)

Φόρτωση του openlanguagedata/flores_plus

https://huggingface.co/datasets/openlanguagedata/flores_plus

In [ ]:
!pip install -q "transformers<5" datasets sentencepiece sacrebleu sacremoses

In [1]:
import gc
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

In [2]:
# Πρόσβαση στο dataset
notebook_login()

In [4]:
# Καλούμε και αποθηκεύουμε το ελληνικό και το αγγλικό devtest.
ell_dataset = load_dataset("openlanguagedata/flores_plus", "ell_Grek", split="devtest")
eng_dataset = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/229 [00:00<?, ?it/s]

ell_Grek.jsonl: 0.00B [00:00, ?B/s]

ell_Grek.jsonl: 0.00B [00:00, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

... (2 outputs left out)

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

## Διερεύνηση του flores

In [5]:
print("Lengths:", len(eng_dataset), len(ell_dataset))
print("Types:", type(eng_dataset), type(ell_dataset))
print(f"Column_named_eng: {eng_dataset.column_names}")
print(f"Column_named_ell: {ell_dataset.column_names}")

Lengths: 1012 1012
Types: <class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'>
Column_named_eng: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split']
Column_named_ell: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split']


In [6]:
print(eng_dataset[0])
print(ell_dataset[0])
print("\n--- First 5 EN ---")
print(*eng_dataset["text"][:5], sep="\n")
print("\n--- First 5 EL ---")
print(*ell_dataset["text"][:5], sep="\n")

{'id': 0, 'iso_639_3': 'eng', 'iso_15924': 'Latn', 'glottocode': 'stan1293', 'variant': '', 'text': '"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.', 'url': 'https://en.wikinews.org/wiki/Toronto_team-led_research_on_Type_1_Diabetes_%27groundbreaking%27', 'domain': 'wikinews', 'topic': 'disease, research, canada', 'has_image': 'no', 'has_hyperlink': 'yes', 'last_updated': '1.0', 'split': 'devtest'}
{'id': 0, 'iso_639_3': 'ell', 'iso_15924': 'Grek', 'glottocode': 'mode1248', 'variant': '', 'text': 'Είπε «επειδή τώρα έχουμε 4 μηνών ποντίκια μη διαβητικά τα οποία κάποτε ήταν διαβητικά».', 'url': 'https://en.wikinews.org/wiki/Toronto_team-led_research_on_Type_1_Diabetes_%27groundbreaking%27', 'domain': 'wikinews', 'topic': 'disease, research, canada', 'has_image': 'no', 'has_hyperlink': 'yes', 'last_updated': '1.0', 'split': 'devtest'}

--- First 5 EN ---
"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
Dr. Eh

## Αποθήκευση των προτάσεων των dataset σε αρχεία txt

In [7]:
with open("Flores_ell.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ell_dataset["text"]) + "\n")

with open("Flores_eng.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(eng_dataset["text"]) + "\n")

# 2.4 Υλοποίηση μεταφράσεων

## Φόρτωση των δεδομένων

In [8]:
with open("Flores_ell.txt", "r", encoding="utf-8") as f:
    ell_dataset = f.read().splitlines()

with open("Flores_eng.txt", "r", encoding="utf-8") as f:
    eng_dataset = f.read().splitlines()

with open("data/test.el", "r", encoding="utf-8") as f:
    test_ell = f.read().splitlines()

with open("data/test.en", "r", encoding="utf-8") as f:
    test_eng = f.read().splitlines()

print("FLORES:", len(ell_dataset), len(eng_dataset))
print("Test set:", len(test_ell), len(test_eng))

FLORES: 1012 1012
Test set: 2000 2000


## 2.4.1 facebook/nllb-200-distilled-600M

In [9]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="ell_Grek")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

### Δοκιμή

(https://huggingface.co/docs/transformers/model_doc/nllb)

In [10]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."
inputs = tokenizer(text, return_tensors="pt").to(model.device)

translated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=30)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

"Good morning, it's Saturday and the weather is hot."

### Εκτέλεση του NLLB-200 στο Flores Dataset

In [11]:
batch_size = 32
model_translations = []

for i in range(0, len(ell_dataset), batch_size):
    batch = ell_dataset[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
    all_translations = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=256)
    batch_preds = tokenizer.batch_decode(all_translations, skip_special_tokens=True)
    model_translations.extend(batch_preds)

print("FLORES translations:", len(model_translations))

with open("facebook_nllb_200_translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(model_translations) + "\n")

FLORES translations: 1012


### Εκτέλεση του NLLB-200 στο Test Set

In [12]:
test_translations = []

for i in range(0, len(test_ell), batch_size):
    batch = test_ell[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
    all_translations = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=256)
    batch_preds = tokenizer.batch_decode(all_translations, skip_special_tokens=True)
    test_translations.extend(batch_preds)

print("Test set translations:", len(test_translations))

with open("facebook_nllb_200_test_translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_translations) + "\n")

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

Test set translations: 2000


## 2.4.2 Helsinki-NLP/opus-mt-grk-en

https://huggingface.co/Helsinki-NLP/opus-mt-grk-en

In [13]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "Helsinki-NLP/opus-mt-grk-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/507k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/441k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/225M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

### Δοκιμή

In [14]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."
inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
translated = model.generate(**inputs)

print(tokenizer.batch_decode(translated, skip_special_tokens=True)[0])

model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

Good morning, it's Saturday and the weather is warm.


### Εκτέλεση του Helsinki-NLP στο Flores Dataset

In [15]:
batch_size = 32
model_translations = []

for i in range(0, len(ell_dataset), batch_size):
    batch = ell_dataset[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    translations = model.generate(**inputs, max_length=256)
    batch_preds = tokenizer.batch_decode(translations, skip_special_tokens=True)
    model_translations.extend(batch_preds)

print("FLORES translations:", len(model_translations))

with open("Helsinki_NLP_Translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(model_translations) + "\n")

FLORES translations: 1012


### Εκτέλεση του Helsinki-NLP στο Test Set

In [16]:
test_translations = []

for i in range(0, len(test_ell), batch_size):
    batch = test_ell[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    translations = model.generate(**inputs, max_length=256)
    batch_preds = tokenizer.batch_decode(translations, skip_special_tokens=True)
    test_translations.extend(batch_preds)

print("Test set translations:", len(test_translations))

with open("Helsinki_NLP_Test_Translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_translations) + "\n")

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

Test set translations: 2000
